# Part 1: Exploratory Data Analysis (EDA)

In this notebook, we'll explore the raw synthetic data generated for the ZomaThon Cart Super Add-On (CSAO) prediction task. We aim to understand data distributions, user ordering behavior, cart sizes, and potential biases in the dataset before moving to feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Data
We load the raw parquet files containing our unaggregated transaction data.

In [ ]:
# Load raw orders
orders_df = pd.read_parquet("../data/raw/orders_exploded_adv.parquet")

# Display basic info
print(f"Total items ordered: {len(orders_df)}")
print(f"Unique orders: {orders_df['order_id'].nunique()}")
orders_df.head()

## 2. Order Distribution Analysis
Let's look at when people are ordering. This helps us understand temporal features.

In [ ]:
# Extract hour of day for order trends
orders_df['hour'] = pd.to_datetime(orders_df['order_time']).dt.hour

plt.figure(figsize=(12, 6))
sns.countplot(data=orders_df, x='hour', palette="viridis")
plt.title('Distribution of Orders by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Items Ordered')
plt.show()

> **Observation**: We can clearly see peak order volumes during Lunch (12-2 PM) and Dinner (7-9 PM) hours. This temporal context is a crucial signal for the recommendation system.

## 3. Cart Size Analysis
How many items are typically in a cart? The CSAO model aims to increase this number.

In [ ]:
# Group by order_id to get cart sizes
cart_sizes = orders_df.groupby('order_id').size().reset_index(name='cart_size')

plt.figure(figsize=(10, 5))
sns.histplot(cart_sizes['cart_size'], bins=15, kde=True, color="coral")
plt.title('Distribution of Cart Sizes')
plt.xlabel('Number of Items in Cart')
plt.ylabel('Frequency')
plt.show()

> **Observation**: The majority of orders consist of 2-4 items. Single-item carts present a huge opportunity for cross-selling complementary add-ons.

## 4. Item Popularity and Category Distribution

In [ ]:
# Load item metadata
items_df = pd.read_parquet("../data/raw/items_adv.parquet")

# Merge with orders to get category info
order_items = orders_df.merge(items_df, on='item_id')

category_counts = order_items['category'].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=category_counts.values, y=category_counts.index, palette="magma")
plt.title('Order Frequency by Item Category')
plt.xlabel('Number of Orders')
plt.ylabel('Category')
plt.show()

## Conclusion
The dataset reflects realistic ordering patterns, heavily skewed toward peak meal times and core categories (Mains/Starters). The feature engineering pipeline must capture this temporal intent (e.g., `is_dinner_time`) and existing cart composition (e.g., `cart_has_main`) to serve highly contextual add-on predictions.